prima fase di conversione

In [1]:
# import json
# import ijson

# with open(r"C:\Users\6060650\Downloads\public_v2.json", "r") as infile, open("public_v2.ndjson", "w") as outfile:
#     for key, value in ijson.kvitems(infile, ''):
#         value["log_id"] = key
#         outfile.write(json.dumps(value) + "\n")

In [2]:
import duckdb
import pandas as pd

con = duckdb.connect()

# Limitiamo a 10.000 righe per velocità
df = con.execute("""
    SELECT *
    FROM read_json_auto('public_v2.ndjson')
    LIMIT 100000
""").df()

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 10 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   referrer   100000 non-null  object
 1   request    100000 non-null  object
 2   method     99256 non-null   object
 3   resource   99256 non-null   object
 4   bytes      100000 non-null  object
 5   response   100000 non-null  object
 6   ip         100000 non-null  object
 7   useragent  100000 non-null  object
 8   timestamp  100000 non-null  object
 9   log_id     100000 non-null  object
dtypes: object(10)
memory usage: 7.6+ MB


In [3]:
# %% [2] - Aggiungiamo pochi attacchi artificiali
attacks = [
    {
        "referrer": "http://search.lib.auth.gr/Search/<script>alert('XSS')</script>",
        "request": "search.lib.auth.gr:80 185.22.12.44 - - [01/Mar/2018:00:00:15 +0200] GET /Search?q=<script>alert('XSS')</script> HTTP/1.1 400 1200 http://search.lib.auth.gr/Search/<script>alert('XSS')</script> Mozilla/5.0",
        "method": "GET",
        "resource": "/Search?q=<script>alert('XSS')</script>",
        "bytes": "1200",
        "response": "400",
        "ip": "185.22.12.44",
        "useragent": "Mozilla/5.0",
        "timestamp": "2018-02-28T22:00:15.000Z",
        "log_id": "attack_xss"
    },
    {
        "referrer": "http://evil.com",
        "request": "search.lib.auth.gr:80 45.77.88.12 - - [01/Mar/2018:00:00:20 +0200] GET /login?user=admin' OR 1=1-- HTTP/1.1 500 800 http://evil.com Mozilla/5.0",
        "method": "GET",
        "resource": "/login?user=admin' OR 1=1--",
        "bytes": "800",
        "response": "500",
        "ip": "45.77.88.12",
        "useragent": "Mozilla/5.0",
        "timestamp": "2018-02-28T22:00:20.000Z",
        "log_id": "attack_sql"
    },
    {
        "referrer": "-",
        "request": "search.lib.auth.gr:80 91.200.12.77 - - [01/Mar/2018:00:00:25 +0200] GET /../../../../etc/passwd HTTP/1.1 403 300 - curl/7.58.0",
        "method": "GET",
        "resource": "/../../../../etc/passwd",
        "bytes": "300",
        "response": "403",
        "ip": "91.200.12.77",
        "useragent": "curl/7.58.0",
        "timestamp": "2018-02-28T22:00:25.000Z",
        "log_id": "attack_traversal"
    },
    {
        "referrer": "-",
        "request": "search.lib.auth.gr:80 203.0.113.45 - - [01/Mar/2018:00:00:30 +0200] GET /wp-admin/install.php HTTP/1.1 404 250 - sqlmap/1.7",
        "method": "GET",
        "resource": "/wp-admin/install.php",
        "bytes": "250",
        "response": "404",
        "ip": "203.0.113.45",
        "useragent": "sqlmap/1.7",
        "timestamp": "2018-02-28T22:00:30.000Z",
        "log_id": "attack_scanner"
    }
]

df = pd.concat([df, pd.DataFrame(attacks)], ignore_index=True)


conversione in tipo timestamp coerente

In [4]:
df["timestamp"] = pd.to_datetime(df["timestamp"])

In [5]:
#lughezza della resource richiesta

df["resource_len"] = df["resource"].str.len()


In [6]:
#da valutare se devo fare un primo controllo anche io sui quartili
from anomalies_detector import has_suspicious_request_len

df = has_suspicious_request_len(df)


c:\Users\6060683\OneDrive - Gruppo Ferrovie Dello Stato\File di MANINI FEDERICO - script\anomalies_detector.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataf["resource_len"] = dataf["resource"].str.len()
c:\Users\6060683\OneDrive - Gruppo Ferrovie Dello Stato\File di MANINI FEDERICO - script\anomalies_detector.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataf.loc[
c:\Users\6060683\OneDrive - Gruppo Ferrovie Dello Stato\File di MANINI FEDERICO - script\anomalies_detector.py:16: SettingWithCo

In [7]:
#troppi special characters
df["special_chars"] = df["request"].str.count(r"[<>\'\"%;(){}]*+@")
df["special_chars"].value_counts()

special_chars
0    95304
1     3956
Name: count, dtype: int64

In [8]:
#directory traversale
df["has_traversal"] = df["request"].str.contains(r"\.\./", regex=True, na=False).astype(int)
df["has_traversal"].value_counts()

has_traversal
0    99259
1        1
Name: count, dtype: int64

In [9]:
# sql injection

df["has_sql_keywords"] = df["request"].str.contains(
    r"union|select|insert|drop|or 1=1|--|admin",
    case=False,
    na=False
).astype(int)

df["has_sql_keywords"].value_counts()

has_sql_keywords
0    99258
1        2
Name: count, dtype: int64

In [10]:
#script injection
df["has_script_tag"] = df["request"].str.contains(
    r"<script>",
    case=False,
    na=False
).astype(int)
df["has_script_tag"].value_counts()

has_script_tag
0    99259
1        1
Name: count, dtype: int64

In [11]:
#possibile scanner da parte di bot
df["scanner_agent"] = df["useragent"].str.contains(
    r"sqlmap|nikto|curl|nmap|dirbuster",
    case=False,
    na=False
).astype(int)
df["useragent"].value_counts()

useragent
Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/64.0.3282.186 Safari/537.36                                                                                     15210
TurnitinBot (https://turnitin.com/robot/crawlerinfo.html)                                                                                                                                                7596
BUbiNG (+http://law.di.unimi.it/BUbiNG.html)                                                                                                                                                             7077
Mozilla/5.0 (Windows NT 6.1; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/64.0.3282.186 Safari/537.36                                                                                       5577
Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:58.0) Gecko/20100101 Firefox/58.0                                                                                        

In [12]:
# status code di errore
df["is_error"] = (df["response"].astype(int) >= 400).astype(int)

df['is_error'].value_counts()

is_error
0    99103
1      157
Name: count, dtype: int64

In [13]:
# numero di query eccessivo
df["num_params"] = df["resource"].str.count("=")
df["num_params"].value_counts()

num_params
0    87127
1    11985
5       97
4       31
3       13
6        5
2        2
Name: count, dtype: int64

In [14]:
# numero di richieste per IP
df["requests_per_ip"] = df.groupby("ip")["ip"].transform("count")


In [15]:
import numpy as np

def entropy(s):
    prob = [float(s.count(c)) / len(s) for c in dict.fromkeys(list(s))]
    entropy = -sum([p * np.log2(p) for p in prob])
    return entropy


df["request_entropy"] = df["request"].apply(lambda x: entropy(str(x)))


In [16]:
df['bytes_num'] = pd.to_numeric(df['bytes'], errors='coerce').fillna(0)
resource_avg_bytes = df.groupby('resource')['bytes_num'].transform('mean')

df['bytes_ratio'] = df['bytes_num'] / (resource_avg_bytes + 1)
df['bytes_ratio'].value_counts()

bytes_ratio
0.998525    16228
0.998527     4551
0.997222     3738
0.997963     2704
0.997230      758
            ...  
0.996678        1
0.999914        1
0.999958        1
0.999902        1
0.999909        1
Name: count, Length: 17105, dtype: int64

In [17]:
df['is_image'] = df['resource'].str.contains(r'\.(jpg|jpeg|png|gif|svg)', case=False, na=False).astype(int)

C:\Users\6060683\AppData\Local\Temp\ipykernel_16084\693450122.py:1: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df['is_image'] = df['resource'].str.contains(r'\.(jpg|jpeg|png|gif|svg)', case=False, na=False).astype(int)


In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 99260 entries, 0 to 100003
Data columns (total 24 columns):
 #   Column                      Non-Null Count  Dtype              
---  ------                      --------------  -----              
 0   referrer                    99260 non-null  object             
 1   request                     99260 non-null  object             
 2   method                      99260 non-null  object             
 3   resource                    99260 non-null  object             
 4   bytes                       99260 non-null  object             
 5   response                    99260 non-null  object             
 6   ip                          99260 non-null  object             
 7   useragent                   99260 non-null  object             
 8   timestamp                   99260 non-null  datetime64[ns, UTC]
 9   log_id                      99260 non-null  object             
 10  resource_len                99260 non-null  int64             

DA QUI IN POI ABBIAMO L'ALGORITMO DI ISOLATION FOREST
1. L'algoritmo non accetta stringhe o date. Dobbiamo isolare solo le colonne numeriche (quelle che hai creato con tanta cura).


In [19]:
from sklearn.ensemble import IsolationForest

# Definiamo le colonne che l'algoritmo deve "studiare"
features = [
    'resource_len', 'has_suspicious_request_len', 'special_chars',
    'has_traversal', 'has_sql_keywords', 'has_script_tag',
    'scanner_agent', 'is_error', 'num_params',
    'requests_per_ip', 'request_entropy', 'bytes_num',
    'bytes_ratio', 'is_image'
]

X = df[features]
X

,resource_len,has_suspicious_request_len,special_chars,has_traversal,has_sql_keywords,has_script_tag,scanner_agent,is_error,num_params,requests_per_ip,request_entropy,bytes_num,bytes_ratio,is_image
0,70,0.0,0,0,0,0,0,0,0,228,5.128159,491,0.997967,0
1,72,0.0,0,0,0,0,0,0,0,1293,5.086639,23282,0.999957,0
2,72,0.0,0,0,0,0,0,0,0,15,5.075418,8952,0.999888,0
3,70,0.0,0,0,0,0,0,0,0,15,5.103453,490,0.997963,0
4,80,0.0,0,0,0,0,0,0,0,15,5.148262,677,0.998525,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99999,80,0.0,0,0,0,0,0,0,0,743,5.187642,677,0.998525,1
100000,39,1.0,0,0,0,1,0,1,1,1,5.028478,1200,0.999167,0
100001,27,1.0,0,0,1,0,0,1,2,1,4.966076,800,0.998752,0
100002,23,1.0,0,1,0,0,1,1,0,1,4.569546,300,0.996678,0


2. Configurazione e Addestramento
L'Isolation Forest funziona creando alberi decisionali casuali: le anomalie vengono "isolate" più velocemente (richiedono meno biforcazioni).


In [20]:
# contamination è la % stimata di attacchi nel dataset (es. 0.1% = 0.001)
model = IsolationForest(n_estimators=100, contamination=0.03, random_state=42)

# Addestramento
df['anomaly_score'] = model.fit_predict(X)

# Risultato: -1 significa Anomalia, 1 significa Traffico Normale

Coding alternativo per fitting del modello:

In [24]:
model = IsolationForest()

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

3. Interpretazione dei risultati
Oltre al flag -1/1, è utilissimo guardare il decision_function, che dà un punteggio continuo: più è basso, più la richiesta è "strana".


In [22]:
# Punteggio grezzo: valori negativi = più anomalo
df['scores'] = model.decision_function(X)

# Vediamo se i tuoi attacchi artificiali sono stati scovati
df[df['log_id'].str.contains('attack')][['log_id', 'anomaly_score', 'scores']]

c:\Users\6060683\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but IsolationForest was fitted without feature names
  warnings.warn(


NotFittedError: This IsolationForest instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.

4. Verifica dei Falsi Positivi
Ordina il dataset per il punteggio più basso per vedere cosa l'algoritmo considera "peggiore" tra i dati reali:

In [ ]:
# Mostra le 10 richieste più sospette del dataset
top_anomalies = df.sort_values(by='scores').head(10)
top_anomalies[['request', 'scores']]

Dato che ho inserito degli attacchi manualmente (attack_xss, attack_sql, ecc.), posso calcolare una Matrice di Confusione semplificata per dimostrare quanto l'algoritmo è efficace nel trovare "l'ago nel pagliaio" che ho nascosto io.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report


# 1. Creiamo una colonna "realtà" (y_true)
# Assumiamo che tutto sia normale (0) tranne i log_id che iniziano con 'attack' (1)
df['is_attack_real'] = df['log_id'].str.contains('attack').astype(int)

# 2. Trasformiamo la predizione dell'Isolation Forest (y_pred)
# L'algoritmo sputa -1 per anomalia e 1 per normale. Lo convertiamo in 1 e 0.
df['is_attack_predicted'] = df['anomaly_score'].apply(lambda x: 1 if x == -1 else 0)

# 3. Generiamo la matrice
cm = confusion_matrix(df['is_attack_real'], df['is_attack_predicted'])

# 4. Visualizziamola graficamente
# plt.figure(figsize=(8, 6))
# sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
#             xticklabels=['Normale', 'Attacco'],
#             yticklabels=['Normale', 'Attacco'])
# plt.xlabel('Predizione Modello')
# plt.ylabel('Realtà (Ground Truth)')
# plt.title('Matrice di Confusione')
# plt.show()

# 5. Report di precisione e recall


In [ ]:
print(classification_report(df['is_attack_real'], df['is_attack_predicted']))

In [ ]:
test_attacks = df[df['log_id'].str.contains('attack')]
test_attacks[['log_id', 'anomaly_score', 'scores']]